In [22]:
import matplotlib.pyplot as plt
from arch import arch_model
import pandas_ta as ta
import pandas as pd 
import numpy as np 
import os
import matplotlib.ticker as mtick


In [8]:
SCALE      = 100       # rescale returns before GARCH fit
WIN        = 180       # rolling window (days)
RSI_LEN    = 20        # RSI period (5-min bars)
BB_LEN     = 20        # Bollinger Band period
RSI_HI     = 60        # overbought threshold  (orig: 70)
RSI_LO     = 40        # oversold threshold    (orig: 30)
TC         = 0.0001    # one-way transaction cost (1 bp)
SLIP       = 0.0001    # slippage per trade (1 bp)
SPLIT_DATE = '2023-01-01'   # train | test boundary
GARCH_P    = 1         # GARCH lag
GARCH_Q    = 3         # ARCH lags
USE_GJR    = True      # True → GJR-GARCH, False → standard GARCH

In [2]:
dd = pd.read_csv('simulated_daily_data.csv')
dd['Date'] = pd.to_datetime(dd['Date'])
dd = dd.set_index('Date')

In [3]:
dd=dd.drop('Unnamed: 7', axis =1)

In [ ]:
id_ = pd.read_csv('simulated_5min_data.csv')
id_['datetime'] = pd.to_datetime(id_['datetime'])
id_ = id_.set_index('datetime')
id_['date'] = pd.to_datetime(id_.index.date)


In [6]:
id_ = id_.drop(columns=[c for c in id_.columns if 'Unnamed' in c])

In [14]:
df = df['2022':]

In [15]:
def predict_volatility(x):
    
    model = arch_model(y=x,p=1,q=3).fit(update_freq=5,disp='off')
    variance_forecast = model.forecast(horizon=1).variance.iloc[-1,0]

    print(x.index[-1])
    
    return variance_forecast

Daily Feature Engineering

In [9]:
dd['lr']  = np.log(dd['Adj Close']).diff()          # log return
dd['rv']  = dd['lr'].rolling(WIN).var()             
dd['lrs'] = dd['lr'] * SCALE     

In [10]:
# Sum of squared 5-min log returns per day → daily realised variance
id_['lr5'] = np.log(id_['close']).diff()
rv5 = (id_['lr5'] ** 2).groupby(id_['date']).sum().rename('rv5')
rv5.index = pd.to_datetime(rv5.index)
dd = dd.join(rv5)
dd['rv_use'] = dd['rv5'].fillna(dd['rv'])  

In [11]:
dd = dd['2022':]
print(f'Daily after 2022: {len(dd):,} rows')
dd[['lr','rv','rv5','rv_use']].dropna().tail()

Daily after 2022: 626 rows


,lr,rv,rv5,rv_use
Date,,,,
2023-09-14,0.011801,0.000393,0.000262,0.000262
2023-09-15,0.002597,0.000385,0.000200,0.000200
2023-09-16,-0.001520,0.000384,0.000061,0.000061
2023-09-17,-0.001284,0.000383,0.000051,0.000051
2023-09-18,0.008261,0.000378,0.000414,0.000414


Volatility Forecast

In [12]:
def fit_garch(x, p=GARCH_P, q=GARCH_Q, gjr=USE_GJR):
    vol = 'GARCH' if not gjr else 'GARCH'
    o   = 1 if gjr else 0            # GJR asymmetry order
    m   = arch_model(x, p=p, o=o, q=q, vol='GARCH')
    r   = m.fit(update_freq=0, disp='off')
    fv  = r.forecast(horizon=1).variance.iloc[-1, 0]
    return fv / (SCALE ** 2) 

In [13]:
dd['pv'] = (
    dd['lrs']
    .rolling(WIN)
    .apply(lambda x: fit_garch(x), raw=True)
)
dd = dd.dropna(subset=['pv', 'rv_use'])
print(f'Rows with GARCH forecast: {len(dd):,}')

Rows with GARCH forecast: 447


Daily Signal Construction

In [ ]:
dd['pp']    = (dd['pv'] - dd['rv_use']) / dd['rv_use']   # normalised 
dd['pp_sd'] = dd['pp'].rolling(WIN).std()

In [16]:
def dsig(row):
    if pd.isna(row['pp_sd']): return np.nan
    if row['pp'] >  row['pp_sd']: return  1.0
    if row['pp'] < -row['pp_sd']: return -1.0
    return np.nan

In [17]:
dd['sig_d'] = dd.apply(dsig, axis=1).shift(1)   # shift → no look-ahead

n_sig = dd['sig_d'].notna().sum()
print(f'Daily signal fires on {n_sig} / {len(dd)} days  ({100*n_sig/len(dd):.1f}%)')
dd['sig_d'].value_counts(dropna=True)

Daily signal fires on 35 / 447 days  (7.8%)


sig_d
1.0    35
Name: count, dtype: int64

Train/Test Split

In [ ]:
train_d = dd[dd.index <  SPLIT_DATE]
test_d  = dd[dd.index >= SPLIT_DATE]

Intraday Signal Construction

In [19]:
df = (
    id_.reset_index()
    .merge(dd[['sig_d']].reset_index(), left_on='date', right_on='Date', how='left')
    .drop(columns=['date', 'Date'])
    .set_index('datetime')
)

In [23]:
df['rsi']  = ta.rsi(df['close'], length=RSI_LEN)
bb         = ta.bbands(df['close'], length=BB_LEN)
df['lbb']  = bb.iloc[:, 0]    # lower band
df['ubb']  = bb.iloc[:, 2]    # upper band

In [24]:
def isig(row):
    if pd.isna(row['rsi']) or pd.isna(row['ubb']): return np.nan
    if row['rsi'] > RSI_HI and row['close'] > row['ubb']: return  1.0
    if row['rsi'] < RSI_LO and row['close'] < row['lbb']: return -1.0
    return np.nan

In [29]:
df['sig_i'] = df.apply(isig, axis=1)


In [25]:
def csig(row):
    if row['sig_d'] ==  1 and row['sig_i'] ==  1: return -1.0  # short
    if row['sig_d'] == -1 and row['sig_i'] == -1: return  1.0  # long
    return np.nan

In [30]:
df['rs'] = df.apply(csig, axis=1)

In [31]:
df['rs'] = df.groupby(pd.Grouper(freq='D'))['rs'].transform(lambda x: x.ffill())


In [32]:
df['rs_prev'] = df['rs'].shift(1)
df['entry']   = (df['rs'] != df['rs_prev']) & df['rs'].notna()

print(f'Total intraday bars : {len(df):,}')
print(f'Bars with position  : {df["rs"].notna().sum():,}')
print(f'Trade entries       : {df["entry"].sum():,}')

Total intraday bars : 177,877
Bars with position  : 4,981
Trade entries       : 31


Return Calc

In [35]:
df['ret']  = np.log(df['close']).diff()

In [36]:
df['next_open']  = df['open'].shift(-1)
df['fill_ret']   = np.log(df['next_open'] / df['close'])   # entry slippage

In [37]:
df['fwd_ret'] = df['ret'].shift(-1)

In [39]:
cost_per_entry = TC + SLIP                    # total cost one-way
df['cost'] = df['entry'] * cost_per_entry * 2  # round-trip on entry bar

df['bar_vol']  = df['ret'].rolling(WIN * 78).std()   # ~180 days of 5-min bars
df['pos_size'] = (df['bar_vol'].mean() / df['bar_vol']).clip(0.2, 3.0)
df['pos_size'] = df['pos_size'].fillna(1.0)
df['strat_ret'] = df['rs'] * df['fwd_ret'] * df['pos_size'] - df['cost']




In [40]:
daily = df.groupby(pd.Grouper(freq='D'))['strat_ret'].sum()
bh    = df.groupby(pd.Grouper(freq='D'))['ret'].sum()   # buy-and-hold benchmark

In [41]:
train_ret = daily[daily.index <  SPLIT_DATE]
test_ret  = daily[daily.index >= SPLIT_DATE]
train_bh  = bh[bh.index <  SPLIT_DATE]
test_bh   = bh[bh.index >= SPLIT_DATE]


In [42]:
print(f'Train days: {len(train_ret)}  |  Test days: {len(test_ret)}')

Train days: 459  |  Test days: 263


Performance Metrics

In [43]:
def metrics(ret_series, label='Strategy', rf=0.0):
    r   = ret_series.dropna()
    ann = 252

    cum  = np.exp(r.cumsum()) - 1
    tot  = cum.iloc[-1]

    ar   = np.exp(r.mean() * ann) - 1
    vol  = r.std() * np.sqrt(ann)

    
    sr   = (r.mean() - rf / ann) / r.std() * np.sqrt(ann) if r.std() > 0 else 0

    wealth  = np.exp(r.cumsum())
    peak    = wealth.cummax()
    dd_ser  = (wealth - peak) / peak
    mdd     = dd_ser.min()

    calmar  = ar / abs(mdd) if mdd != 0 else np.nan

    active  = r[r != 0]
    wr      = (active > 0).mean() if len(active) else np.nan

    sig = df['rs'].dropna()
    chg = sig != sig.shift()
    run_ids  = chg.cumsum()
    avg_hold = (~chg).groupby(run_ids).sum().mean()

    print(f'\n── {label} ──')
    print(f'  Total return      : {tot:+.2%}')
    print(f'  Ann. return       : {ar:+.2%}')
    print(f'  Ann. volatility   : {vol:.2%}')
    print(f'  Sharpe ratio      : {sr:.2f}')
    print(f'  Max drawdown      : {mdd:.2%}')
    print(f'  Calmar ratio      : {calmar:.2f}')
    print(f'  Win rate (daily)  : {wr:.2%}')
    print(f'  Avg hold (bars)   : {avg_hold:.1f}')
    return {'total':tot,'ann_ret':ar,'vol':vol,'sharpe':sr,'mdd':mdd,'wr':wr}



In [44]:
print("TRAIN PERIOD")
m_train = metrics(train_ret, 'Strategy (train)')
m_bh_tr = metrics(train_bh,  'Buy & Hold (train)')

print(' TEST PERIOD ')
m_test  = metrics(test_ret,  'Strategy (test)')
m_bh_te = metrics(test_bh,   'Buy & Hold (test)')

TRAIN PERIOD

── Strategy (train) ──
  Total return      : +0.62%
  Ann. return       : +0.34%
  Ann. volatility   : 0.38%
  Sharpe ratio      : 0.89
  Max drawdown      : -0.09%
  Calmar ratio      : 3.70
  Win rate (daily)  : 66.67%
  Avg hold (bars)   : 4980.0

── Buy & Hold (train) ──
  Total return      : -60.15%
  Ann. return       : -39.66%
  Ann. volatility   : 52.76%
  Sharpe ratio      : -0.96
  Max drawdown      : -76.87%
  Calmar ratio      : -0.52
  Win rate (daily)  : 49.67%
  Avg hold (bars)   : 4980.0
 TEST PERIOD 

── Strategy (test) ──
  Total return      : +5.01%
  Ann. return       : +4.80%
  Ann. volatility   : 7.16%
  Sharpe ratio      : 0.65
  Max drawdown      : -6.63%
  Calmar ratio      : 0.72
  Win rate (daily)  : 50.00%
  Avg hold (bars)   : 4980.0

── Buy & Hold (test) ──
  Total return      : +64.03%
  Ann. return       : +60.67%
  Ann. volatility   : 37.66%
  Sharpe ratio      : 1.26
  Max drawdown      : -20.18%
  Calmar ratio      : 3.01
  Win rate (dai

In [45]:
summary = pd.DataFrame({
    'Metric'      : ['Total Return','Ann. Return','Ann. Vol','Sharpe','Max Drawdown','Win Rate'],
    'Train Strat' : [f"{m_train['total']:+.2%}", f"{m_train['ann_ret']:+.2%}",
                     f"{m_train['vol']:.2%}",    f"{m_train['sharpe']:.2f}",
                     f"{m_train['mdd']:.2%}",    f"{m_train['wr']:.2%}"],
    'Train B&H'   : [f"{m_bh_tr['total']:+.2%}", f"{m_bh_tr['ann_ret']:+.2%}",
                     f"{m_bh_tr['vol']:.2%}",    f"{m_bh_tr['sharpe']:.2f}",
                     f"{m_bh_tr['mdd']:.2%}",    f"{m_bh_tr['wr']:.2%}"],
    'Test Strat'  : [f"{m_test['total']:+.2%}",  f"{m_test['ann_ret']:+.2%}",
                     f"{m_test['vol']:.2%}",     f"{m_test['sharpe']:.2f}",
                     f"{m_test['mdd']:.2%}",     f"{m_test['wr']:.2%}"],
    'Test B&H'    : [f"{m_bh_te['total']:+.2%}", f"{m_bh_te['ann_ret']:+.2%}",
                     f"{m_bh_te['vol']:.2%}",    f"{m_bh_te['sharpe']:.2f}",
                     f"{m_bh_te['mdd']:.2%}",    f"{m_bh_te['wr']:.2%}"],
})
summary.set_index('Metric', inplace=True)
print(summary.to_string())

             Train Strat Train B&H Test Strat Test B&H
Metric                                                
Total Return      +0.62%   -60.15%     +5.01%  +64.03%
Ann. Return       +0.34%   -39.66%     +4.80%  +60.67%
Ann. Vol           0.38%    52.76%      7.16%   37.66%
Sharpe              0.89     -0.96       0.65     1.26
Max Drawdown      -0.09%   -76.87%     -6.63%  -20.18%
Win Rate          66.67%    49.67%     50.00%   49.43%
